In [ ]:
lat between -5 and 5
lon between 190 and 240  # In degrees east (equivalent to 170°W to 120°W)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Filter to Niño 3.4 region
def filter_nino34(df):
    return df[
        (df['lat'] >= -5) & (df['lat'] <= 5) &
        (df['lon'] >= 190) & (df['lon'] <= 240)
    ]

# 2. Monthly average SST in region
def compute_monthly_avg(df):
    df['time'] = pd.to_datetime(df['time'])
    df['month'] = df['time'].dt.to_period('M')
    monthly = df.groupby('month')['sst'].mean().reset_index()
    monthly['time'] = monthly['month'].dt.to_timestamp()
    return monthly[['time', 'sst']]

# 3. Climatology from 1991–2020
def compute_climatology(df):
    df['month_num'] = df['time'].dt.month
    base = df[(df['time'].dt.year >= 1991) & (df['time'].dt.year <= 2020)]
    climatology = base.groupby('month_num')['sst'].mean()
    return climatology

# 4. Compute anomalies
def compute_anomalies(df, climatology):
    df['month_num'] = df['time'].dt.month
    df['anomaly'] = df.apply(lambda row: row['sst'] - climatology.loc[row['month_num']], axis=1)
    return df

# 5. Compute ONI (3-month rolling mean)
def compute_oni(df):
    df['oni'] = df['anomaly'].rolling(window=3, center=True).mean()
    return df

# 6. Plotting
def plot_results(df):
    plt.figure(figsize=(12, 5))
    plt.plot(df['time'], df['oni'], label='ONI (3-month running mean)', color='red')
    plt.axhline(0.5, color='gray', linestyle='--', label='El Niño Threshold')
    plt.axhline(-0.5, color='gray', linestyle='--', label='La Niña Threshold')
    plt.title('Oceanic Niño Index (ONI)')
    plt.ylabel('SST Anomaly (°C)')
    plt.xlabel('Time')
    plt.grid(True, linestyle=':')
    plt.legend()
    plt.tight_layout()
    plt.show()

# MAIN PIPELINE
def calculate_oni_index(df):
    df_nino34 = filter_nino34(df)
    monthly_avg = compute_monthly_avg(df_nino34)
    climatology = compute_climatology(monthly_avg)
    anomalies = compute_anomalies(monthly_avg, climatology)
    oni_df = compute_oni(anomalies)
    return oni_df

# Example usage
# oni_df = calculate_oni_index(df)
# plot_results(oni_df)
